# Earnings Quality & Accrual Analysis

*A CFA Level 1 Deep Dive into Detecting Earnings Manipulation*

---

This notebook explores the concept of **earnings quality** — the degree to which reported
earnings faithfully represent economic reality. We develop quantitative tools for assessing
earnings quality, including the **accrual anomaly**, **accrual decomposition**, and the
**Beneish M-Score** model for detecting earnings manipulation.

**Learning Objectives:**
- Understand the spectrum of earnings quality from conservative to fraudulent reporting
- Compute and interpret total accruals and their components
- Identify common revenue and expense manipulation techniques
- Implement the Beneish M-Score from scratch and classify potential manipulators
- Recognize red flags in financial statements that signal low earnings quality

**Prerequisites:** Basic understanding of financial statements (income statement, balance sheet, cash flow statement) and fundamental accounting concepts.

**Prerequisites:** Introduction to Financial Statements, Income Statement Analysis, Balance Sheet & Working Capital, Cash Flow Statement Analysis.

**Outline**
1. What is earnings quality and why it matters
2. Setup
3. The accrual anomaly — Sloan (1996)
4. Accrual decomposition — balance sheet vs cash flow approach
5. Revenue manipulation techniques
6. Expense manipulation techniques
7. The Beneish M-Score model
8. M-Score classification and portfolio analysis
9. Red flags checklist
10. The auditor's role and limitations
11. References

## 1. What Is Earnings Quality and Why It Matters

### 1.1 Defining Earnings Quality

Earnings quality refers to the ability of reported earnings to **reflect the company's true
economic performance**, to be **sustainable and repeatable**, and to serve as a **reliable
basis for forecasting future cash flows**. High-quality earnings are those that:

1. **Arise from core, recurring operations** rather than one-time gains or accounting adjustments
2. **Are backed by actual cash flows** rather than aggressive accrual assumptions
3. **Result from conservative accounting choices** that do not overstate economic value
4. **Are transparent and well-disclosed** so that analysts can make informed judgments

The concept matters because virtually all equity valuation models — discounted cash flow,
residual income, price-to-earnings multiples — depend on the assumption that reported
earnings approximate economic reality. When that assumption fails, investors misprice
securities and capital is misallocated.

### 1.2 The Earnings Quality Spectrum

Accounting standards (IFRS and US GAAP) provide management with significant discretion in
how transactions are recognized, measured, and disclosed. This discretion exists for good
reason — management has private information about the business that rigid rules cannot
capture. However, it also creates a spectrum of reporting quality:

| Level | Description | Examples |
|-------|-------------|----------|
| **Conservative** | Understates economic performance; builds hidden reserves | Accelerated depreciation when straight-line is appropriate; early recognition of contingent liabilities |
| **Neutral / Faithful** | Accurately reflects economic reality within the bounds of accounting standards | Estimates that are unbiased and well-supported by evidence |
| **Aggressive** | Overstates current performance within the technical boundaries of GAAP/IFRS | Extending useful lives of assets; delaying impairments; capitalizing costs that should be expensed |
| **Fraudulent** | Deliberately violates accounting standards to deceive users | Fabricating revenue; hiding liabilities; falsifying documents |

> **Key Concept:** The boundary between "aggressive" and "fraudulent" accounting is not always
> clear-cut. Aggressive accounting pushes the limits of what standards allow but remains
> technically compliant. Fraud involves intentional misstatement or omission of material facts.
> However, many fraud cases start as aggressive accounting that gradually crosses the line.

### 1.3 Why Management Manipulates Earnings

Understanding the **incentives** behind earnings management is essential for identifying it:

- **Compensation:** Executive bonuses and stock options are often tied to earnings targets,
  creating direct financial incentives to inflate reported profits
- **Debt covenants:** Loan agreements frequently include financial ratio thresholds (e.g.,
  debt-to-EBITDA, interest coverage). Violating these covenants can trigger default provisions
- **Stock price:** Managers who hold equity or face pressure from investors have incentives to
  meet or beat analyst expectations, since even small misses can cause significant price declines
- **Job security:** Managers who consistently miss targets face termination risk, creating
  survival-driven manipulation
- **Regulatory thresholds:** Banks must maintain capital adequacy ratios; utilities must justify
  rate increases based on reported costs
- **Tax minimization:** In some jurisdictions, tax and financial reporting are linked, creating
  incentives to understate income

> **CFA Exam Tip:** The CFA curriculum emphasizes that analysts should always consider
> management's incentive structure when assessing earnings quality. Look at compensation
> plans, debt covenants, and recent management changes as clues to potential manipulation.

### 1.4 Accrual Accounting and the Root of the Problem

Under accrual accounting, revenue is recognized when earned and expenses when incurred,
regardless of cash timing. This creates a fundamental gap between **reported earnings** and
**cash flows from operations (CFO)**:

$$\text{Net Income} = \text{Cash Flow from Operations} + \text{Total Accruals}$$

Or equivalently:

$$\text{Total Accruals} = \text{Net Income} - \text{CFO}$$

Accruals exist because economic events do not always coincide with cash transactions.
Examples include:

- **Revenue accruals:** A company delivers goods on credit, recording revenue and accounts
  receivable before cash is collected
- **Expense accruals:** A company receives services but has not yet paid the invoice,
  recording an expense and accounts payable
- **Depreciation:** A long-lived asset's cost is allocated over its useful life, reducing
  reported income without any current cash outflow
- **Provisions:** Estimated future obligations (warranties, litigation) are recorded as
  current expenses

While accruals are necessary and proper under GAAP/IFRS, they are also the primary
vehicle through which management exercises discretion — and the primary target for
manipulation.

> **Key Concept:** Cash flows are harder to manipulate than accruals because they involve
> actual transactions with third parties (banks, customers, suppliers). This is why large
> and persistent discrepancies between net income and CFO are a critical red flag for
> earnings quality analysts.

## 2. Setup

We configure the computational environment, set a reproducible random seed, and define
color constants used throughout the notebook's visualizations.

We use synthetic financial data throughout this notebook to demonstrate each concept with known ground truth. In practice, analysts would apply these same techniques to real company filings sourced from SEC EDGAR, Bloomberg, or Capital IQ.

In [ ]:
%matplotlib inline
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# Numerical tolerances
ATOL = 1e-8
RTOL = 1e-6

# Visualization palette
PRIMARY   = "steelblue"
SECONDARY = "coral"
TERTIARY  = "seagreen"
ACCENT    = "gold"

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size":      12,
    "axes.grid":      True,
    "grid.alpha":     0.3,
})

print("Environment ready.")

## 3. The Accrual Anomaly

### 3.1 Theoretical Foundation: Sloan (1996)

One of the most influential findings in empirical accounting research is the **accrual
anomaly**, documented by Richard Sloan in his 1996 paper *"Do Stock Prices Fully Reflect
Information in Accruals and Cash Flows about Future Earnings?"*

Sloan's key insight was that earnings can be decomposed into two components with very
different persistence properties:

$$\text{Earnings}_t = \text{Cash Flow Component}_t + \text{Accrual Component}_t$$

**The persistence finding:** The cash flow component of earnings is significantly more
persistent (predictive of future earnings) than the accrual component. Formally:

$$E[\text{Earnings}_{t+1}] = \alpha + \beta_1 \cdot \text{CFO}_t + \beta_2 \cdot \text{Accruals}_t$$

Empirically, $\beta_1 \approx 0.85$ while $\beta_2 \approx 0.65$. This means a dollar of
earnings backed by cash flows is worth more for forecasting purposes than a dollar of
earnings driven by accruals.

### 3.2 Why Accruals Mean-Revert

The lower persistence of accruals has an economic explanation rooted in the **reversing
nature of accrual accounting**:

1. **Working capital accruals reverse quickly.** If a company aggressively recognizes revenue
   by stuffing the distribution channel (boosting accounts receivable), those receivables must
   eventually be collected or written off. The boost to current earnings is offset by a drag
   on future earnings.

2. **Long-term accruals reverse more slowly but inevitably.** If a company capitalizes costs
   that should be expensed (inflating assets), those assets must eventually be depreciated or
   impaired, dragging down future earnings.

3. **Estimation error in accruals.** Many accruals involve estimates (bad debt provisions,
   warranty reserves, pension obligations). These estimates are inherently less reliable than
   actual cash transactions, and estimation errors tend to correct over time.

> **Key Concept:** The accrual anomaly implies that companies with high positive accruals
> (earnings significantly exceed CFO) tend to experience **earnings declines** in subsequent
> periods, while companies with low or negative accruals (CFO exceeds earnings) tend to
> experience **earnings improvements**. This is one of the most robust anomalies in finance.

### 3.3 Total Accruals Formula

Total accruals can be computed directly:

$$\text{Total Accruals} = \text{Net Income} - \text{Cash Flow from Operations}$$

When scaled by total assets for cross-sectional comparability:

$$\text{Accrual Ratio} = \frac{\text{Net Income} - \text{CFO}}{\text{Average Total Assets}}$$

An accrual ratio significantly above zero suggests earnings are being driven by accrual
assumptions rather than cash generation, warranting closer scrutiny.

> **CFA Exam Tip:** On the CFA exam, you may be asked to compute total accruals using either
> the cash flow statement approach (NI - CFO) or the balance sheet approach (changes in
> working capital accounts minus depreciation). Both should yield the same result, but the
> cash flow statement approach is considered more reliable because it is less susceptible
> to classification manipulation.

### Interpreting the accrual ratio

| Accrual Ratio | Interpretation |
|:---:|:---|
| **< -5%** | Very conservative — cash flow significantly exceeds reported earnings |
| **-5% to +5%** | Normal range — accruals are a modest fraction of assets |
| **+5% to +10%** | Elevated — earnings increasingly depend on estimates and assumptions |
| **> +10%** | Warning zone — investigate whether accruals are justified by business circumstances |

These thresholds are approximate and vary by industry. Capital-intensive industries with large depreciation charges may have structurally different accrual profiles than asset-light service businesses.

> **Common Mistake:** A high accrual ratio does not *prove* manipulation. A rapidly growing company legitimately building receivables and inventory will naturally have positive accruals. The red flag is when high accruals are *not* explained by underlying business growth — or when accruals are persistently high year after year without corresponding cash flow materialising.

In [ ]:
# ------------------------------------------------------------------
# Generate 10-year synthetic data for a single company
# ------------------------------------------------------------------
years = np.arange(2014, 2024)
n_years = len(years)

# Base revenue growing at ~5% per year with noise
base_revenue = 1000 * (1.05 ** np.arange(n_years))
revenue = base_revenue + rng.normal(0, 30, n_years)

# CFO is a noisy fraction of revenue (healthy company: CFO ~ 12-18% of revenue)
cfo = revenue * (0.15 + rng.normal(0, 0.02, n_years))

# Net income includes accrual manipulation: gradually increasing aggressiveness
manipulation = np.linspace(0, 0.04, n_years) * revenue  # growing accrual inflation
net_income = cfo + manipulation + rng.normal(0, 10, n_years)

# Total assets
total_assets = revenue * (1.8 + rng.normal(0, 0.05, n_years))
avg_assets = (total_assets[:-1] + total_assets[1:]) / 2

# Compute accruals
total_accruals = net_income - cfo
accrual_ratio = total_accruals[1:] / avg_assets  # skip first year (no avg assets)

# Next-year earnings change (scaled)
earnings_change = np.diff(net_income) / avg_assets

print(f"{'Year':<6} {'Revenue':>10} {'NI':>10} {'CFO':>10} {'Accruals':>10} {'Accrual %':>10}")
print("-" * 58)
for i, y in enumerate(years):
    pct = f"{total_accruals[i]/total_assets[i]*100:.1f}%" if total_assets[i] != 0 else "N/A"
    print(f"{y:<6} {revenue[i]:>10.1f} {net_income[i]:>10.1f} {cfo[i]:>10.1f} {total_accruals[i]:>10.1f} {pct:>10}")

The table above shows a crucial pattern: net income and CFO do not move in lockstep. In some years, net income exceeds CFO (positive accruals — the company is recognising revenue it hasn't collected in cash), while in others CFO exceeds net income (negative accruals — cash is coming in faster than revenue is being recognised).

The accrual percentage column (accruals as a fraction of total assets) provides the key signal. Consistently high positive accrual ratios (above 5-10%) are a warning sign — they suggest that reported earnings are increasingly driven by accounting assumptions rather than cash generation.

> **CFA Exam Tip:** When analysing a company, always compare the trend in net income to the trend in CFO. If net income is growing but CFO is flat or declining, the earnings growth is being driven by accruals — which, as Sloan demonstrated, tends to be unsustainable.

In [ ]:
# ------------------------------------------------------------------
# Cross-sectional illustration: 80 synthetic companies
# ------------------------------------------------------------------
n_companies = 80

# Generate company-level data
rev_cs = rng.uniform(500, 5000, n_companies)
cfo_cs = rev_cs * rng.uniform(0.08, 0.20, n_companies)
assets_cs = rev_cs * rng.uniform(1.5, 2.5, n_companies)

# Accruals: mixture of honest and aggressive firms
is_aggressive = rng.random(n_companies) < 0.3
accrual_noise = rng.normal(0, 0.02, n_companies)
accrual_pct = np.where(is_aggressive, 0.06 + accrual_noise, 0.01 + accrual_noise)
ni_cs = cfo_cs + accrual_pct * assets_cs

accruals_cs = (ni_cs - cfo_cs) / assets_cs

# Next-year earnings change: negatively correlated with accruals (the anomaly)
next_year_change = -0.5 * accruals_cs + rng.normal(0, 0.03, n_companies)

# Scatter plot
fig, ax = plt.subplots()
colors = np.where(is_aggressive, SECONDARY, PRIMARY)
ax.scatter(accruals_cs, next_year_change, c=colors, alpha=0.7, edgecolors="white", s=60)

# Regression line
slope, intercept, r_value, p_value, _ = stats.linregress(accruals_cs, next_year_change)
x_fit = np.linspace(accruals_cs.min(), accruals_cs.max(), 100)
ax.plot(x_fit, slope * x_fit + intercept, color="black", linewidth=2,
        label=f"OLS: slope={slope:.2f}, R2={r_value**2:.2f}")

ax.set_xlabel("Accrual Ratio (Current Year)")
ax.set_ylabel("Earnings Change / Assets (Next Year)")
ax.set_title("The Accrual Anomaly: High Accruals Predict Earnings Declines")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)

# Custom legend entries
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=PRIMARY, markersize=8, label='Normal firms'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=SECONDARY, markersize=8, label='Aggressive firms'),
    Line2D([0], [0], color='black', linewidth=2, label=f'OLS: slope={slope:.2f}, R2={r_value**2:.2f}'),
]
ax.legend(handles=legend_elements, loc="upper right")
plt.tight_layout()
plt.show()

print(f"\nRegression: next_year_change = {intercept:.4f} + {slope:.4f} * accrual_ratio")
print(f"p-value for slope: {p_value:.6f}")
print(f"Interpretation: Higher accruals predict lower future earnings changes.")

The cross-sectional scatter plot confirms Sloan's finding at the portfolio level: companies with high accrual ratios (right side) tend to experience *negative* earnings changes in the following year, while companies with low or negative accrual ratios (left side) tend to see earnings *improve*.

The regression line slopes downward — a statistically significant negative relationship between current accruals and future earnings changes. This is the accrual anomaly in action: the market prices in current earnings without fully adjusting for the lower persistence of the accrual component, creating a predictable pattern of future earnings reversals.

> **Key Concept:** The colour coding separates aggressive reporters (red) from conservative ones (blue). Notice that the aggressive firms cluster in the upper-right quadrant (high accruals, positive current earnings change) — their inflated earnings look good today but tend to reverse. This visual pattern is exactly what the Beneish M-Score (Section 7) attempts to capture systematically.

The practical investment implication is the **accruals-based trading strategy**: go long low-accrual firms (high earnings quality) and short high-accrual firms (low earnings quality). While this strategy's alpha has diminished since Sloan's original publication — a common finding in anomaly research — the underlying economic logic remains sound for fundamental analysis.

## 4. Accrual Decomposition

### 4.1 Two Approaches to Measuring Accruals

There are two standard approaches to computing total accruals, each with advantages and
limitations.

#### Balance Sheet Approach

The balance sheet approach derives accruals from changes in balance sheet accounts:

$$\text{Total Accruals}_{BS} = \Delta CA - \Delta Cash - \Delta CL + \Delta STD + \Delta TP - Dep$$

Where:
- $\Delta CA$ = Change in current assets
- $\Delta Cash$ = Change in cash and cash equivalents
- $\Delta CL$ = Change in current liabilities
- $\Delta STD$ = Change in short-term debt (included in current liabilities)
- $\Delta TP$ = Change in income taxes payable
- $Dep$ = Depreciation and amortization expense

**Advantage:** Can be computed from the balance sheet alone, useful for historical periods
where detailed cash flow statements may not be available.

**Disadvantage:** Contaminated by non-operating events such as mergers, acquisitions, and
discontinued operations, which change balance sheet accounts without reflecting operating
accruals.

#### Cash Flow Statement Approach

The cash flow statement approach is simpler and more reliable:

$$\text{Total Accruals}_{CF} = \text{Net Income} - \text{CFO}$$

**Advantage:** Directly captures the gap between accounting earnings and cash generation.
Less susceptible to classification issues.

**Disadvantage:** Requires a reliable cash flow statement. The classification of items
between operating, investing, and financing activities can vary across companies.

> **Common Mistake:** Students often confuse total accruals with working capital accruals.
> Working capital accruals are a subset of total accruals — they capture only the short-term
> timing differences (receivables, payables, inventory). Total accruals also include
> long-term items like depreciation and amortization, which represent the allocation of
> past capital expenditures.

### 4.2 Discretionary vs. Non-Discretionary Accruals

A critical distinction in accrual analysis is between:

- **Non-discretionary accruals:** The "normal" level of accruals expected given the company's
  business activities. A growing company naturally has increasing receivables and inventory.
  These accruals reflect legitimate economic activity.

- **Discretionary accruals:** The **abnormal** component of accruals that cannot be explained
  by the company's underlying economics. These represent management's accounting choices and
  are the focus of earnings quality analysis.

$$\text{Total Accruals} = \text{Non-Discretionary Accruals} + \text{Discretionary Accruals}$$

### 4.3 The Jones Model (1991)

The most influential model for decomposing accruals is the **Jones Model**, which estimates
non-discretionary accruals as a function of two variables:

$$\frac{TA_{i,t}}{A_{i,t-1}} = \alpha_1 \frac{1}{A_{i,t-1}} + \alpha_2 \frac{\Delta REV_{i,t}}{A_{i,t-1}} + \alpha_3 \frac{PPE_{i,t}}{A_{i,t-1}} + \varepsilon_{i,t}$$

Where:
- $TA_{i,t}$ = Total accruals for firm $i$ in year $t$
- $A_{i,t-1}$ = Lagged total assets (scaling variable)
- $\Delta REV_{i,t}$ = Change in revenue
- $PPE_{i,t}$ = Gross property, plant, and equipment

**Economic intuition:**
- **Revenue change** captures the expected impact of business growth on working capital
  accruals. Higher revenue growth naturally leads to higher receivables and inventory.
- **PPE** captures the expected level of depreciation. More fixed assets mean more
  depreciation expense, which is a large non-discretionary accrual.

The **residual** $\varepsilon_{i,t}$ from this regression is the estimate of discretionary
accruals — the portion of total accruals that cannot be explained by normal business
activity.

> **Key Concept:** Large positive discretionary accruals suggest that management is using
> accounting choices to inflate earnings beyond what the company's operations justify. Large
> negative discretionary accruals may indicate "big bath" accounting, where management
> intentionally depresses current earnings to create reserves for future periods.

### 4.4 Modified Jones Model

Dechow, Sloan, and Sweeney (1995) proposed a modification that adjusts the revenue change
for the change in receivables:

$$\frac{TA_{i,t}}{A_{i,t-1}} = \alpha_1 \frac{1}{A_{i,t-1}} + \alpha_2 \frac{\Delta REV_{i,t} - \Delta REC_{i,t}}{A_{i,t-1}} + \alpha_3 \frac{PPE_{i,t}}{A_{i,t-1}} + \varepsilon_{i,t}$$

The logic is that revenue increases accompanied by corresponding receivable increases may
reflect credit sales rather than genuine economic growth. By subtracting the receivable
change, the model better isolates the non-discretionary component.

> **CFA Exam Tip:** You are unlikely to be asked to estimate the Jones model on the CFA exam,
> but you should understand the concept: total accruals can be split into an expected
> (non-discretionary) component driven by business fundamentals and an unexpected
> (discretionary) component that reflects management's accounting choices.

### Practical limitations of the Jones model

While the Jones model is foundational, analysts should be aware of its limitations:

1. **Industry specificity:** The model should be estimated within a single industry-year cohort. Mixing industries introduces noise because "normal" accruals vary dramatically (e.g., construction vs software).

2. **Sample size:** Cross-sectional estimation requires a sufficient number of firms. With fewer than 20 firms per industry-year, parameter estimates become unreliable.

3. **Assumption of linearity:** The model assumes a linear relationship between accruals and its determinants. In practice, the relationship may be non-linear, especially during periods of rapid growth or contraction.

4. **Survivorship bias:** Studies using the Jones model typically include only surviving firms, potentially excluding companies that were delisted due to fraud — precisely the firms of greatest interest.

> **Key Concept:** Despite these limitations, the Jones model framework remains the starting point for most academic and professional earnings quality analysis. Its core insight — that accruals can be decomposed into an expected component (driven by business fundamentals) and an unexpected component (potentially driven by manipulation) — is conceptually powerful even when the specific model estimates are imprecise.

In [ ]:
# ------------------------------------------------------------------
# Demonstrate both accrual computation approaches
# ------------------------------------------------------------------

# Generate synthetic balance sheet data for one company over 5 years
years_bs = np.arange(2019, 2024)
n = len(years_bs)

# Balance sheet items (in millions)
current_assets = np.array([450, 480, 520, 590, 640])
cash           = np.array([80,  85,  90,  95,  100])
current_liab   = np.array([300, 310, 330, 350, 370])
short_term_debt= np.array([50,  55,  50,  60,  65])
tax_payable    = np.array([20,  22,  25,  23,  28])
depreciation   = np.array([0,   40,  42,  45,  48])  # annual dep expense

# Income statement
net_inc = np.array([0, 120, 135, 155, 140])
cfo_bs  = np.array([0, 100, 115, 120, 130])

# Balance Sheet Approach (years 1-4, need prior year)
delta_CA   = np.diff(current_assets)
delta_cash = np.diff(cash)
delta_CL   = np.diff(current_liab)
delta_STD  = np.diff(short_term_debt)
delta_TP   = np.diff(tax_payable)
dep        = depreciation[1:]

accruals_bs = delta_CA - delta_cash - delta_CL + delta_STD + delta_TP - dep

# Cash Flow Statement Approach
accruals_cf = net_inc[1:] - cfo_bs[1:]

print("Accrual Decomposition: Balance Sheet vs Cash Flow Approach")
print("=" * 65)
print(f"{'Year':<6} {'BS Accruals':>12} {'CF Accruals':>12} {'Difference':>12}")
print("-" * 42)
for i, y in enumerate(years_bs[1:]):
    print(f"{y:<6} {accruals_bs[i]:>12.1f} {accruals_cf[i]:>12.1f} {accruals_bs[i]-accruals_cf[i]:>12.1f}")

print("\nNote: Differences arise because the BS approach approximates accruals")
print("from balance sheet changes, while CF approach uses actual cash flow data.")

The two approaches produce similar but not identical results. The differences arise because the balance sheet approach *approximates* accruals from changes in working capital accounts, while the cash flow approach directly uses the difference between reported earnings and operating cash flow.

In practice, the cash flow approach is preferred because:
1. It is simpler and less prone to classification errors.
2. Balance sheet changes can be distorted by acquisitions, divestitures, and foreign currency translation.
3. The cash flow statement is audited, providing a reliable CFO figure.

> **Common Mistake:** The balance sheet approach can give misleading results when a company makes an acquisition during the year. The acquired company's assets and liabilities appear as large balance sheet changes that have nothing to do with accrual accounting. The cash flow approach is unaffected because operating cash flow excludes acquisition-related cash flows (they appear in investing activities).

In [ ]:
# ------------------------------------------------------------------
# Jones Model: cross-sectional estimation with synthetic data
# ------------------------------------------------------------------
n_firms = 60

# Synthetic firm data (scaled by lagged assets)
lagged_assets = rng.uniform(500, 3000, n_firms)
delta_rev = rng.normal(100, 80, n_firms)
ppe = lagged_assets * rng.uniform(0.3, 0.7, n_firms)

# True non-discretionary accruals (Jones model)
alpha1, alpha2, alpha3 = 10.0, 0.05, -0.04
nda = alpha1 * (1.0 / lagged_assets) + alpha2 * (delta_rev / lagged_assets) + alpha3 * (ppe / lagged_assets)

# Add discretionary accruals (some firms manipulate)
is_manip = rng.random(n_firms) < 0.25
discretionary = np.where(is_manip, rng.uniform(0.03, 0.08, n_firms), rng.normal(0, 0.01, n_firms))
total_accruals_scaled = nda + discretionary

# OLS estimation of Jones model
X = np.column_stack([
    1.0 / lagged_assets,
    delta_rev / lagged_assets,
    ppe / lagged_assets,
])
# Add intercept
X_with_const = np.column_stack([np.ones(n_firms), X])
y = total_accruals_scaled

# OLS: beta = (X'X)^{-1} X'y
beta = np.linalg.lstsq(X_with_const, y, rcond=None)[0]
fitted = X_with_const @ beta
residuals = y - fitted  # discretionary accruals estimate

print("Jones Model OLS Estimation")
print("=" * 50)
print(f"Intercept:      {beta[0]:.6f}")
print(f"1/Assets:       {beta[1]:.6f}  (true: {alpha1:.1f})")
print(f"DeltaRev/A:     {beta[2]:.6f}  (true: {alpha2:.2f})")
print(f"PPE/A:          {beta[3]:.6f}  (true: {alpha3:.2f})")
print(f"\nR-squared:      {1 - np.var(residuals)/np.var(y):.4f}")

# Plot discretionary accruals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_jones = np.where(is_manip, SECONDARY, PRIMARY)
axes[0].scatter(fitted, residuals, c=colors_jones, alpha=0.7, edgecolors="white", s=50)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_xlabel("Fitted (Non-Discretionary Accruals)")
axes[0].set_ylabel("Residual (Discretionary Accruals)")
axes[0].set_title("Jones Model: Discretionary Accruals")

axes[1].hist(residuals[~is_manip], bins=15, alpha=0.7, color=PRIMARY, label="Normal firms", density=True)
axes[1].hist(residuals[is_manip], bins=10, alpha=0.7, color=SECONDARY, label="Manipulators", density=True)
axes[1].set_xlabel("Discretionary Accruals (Residual)")
axes[1].set_ylabel("Density")
axes[1].set_title("Distribution of Discretionary Accruals")
axes[1].legend()

plt.tight_layout()
plt.show()

mean_normal = residuals[~is_manip].mean()
mean_manip = residuals[is_manip].mean()
print(f"\nMean discretionary accruals -- Normal: {mean_normal:.4f}, Manipulators: {mean_manip:.4f}")

The Jones model regression estimates the "normal" level of accruals that a company should have, given its revenue growth and asset base. The **residuals** — the difference between actual and predicted accruals — represent the *discretionary* component that management controls.

The right panel is the key diagnostic: manipulator firms (red) cluster at higher residual values than clean firms (blue). Their actual accruals systematically exceed what the model predicts as normal, suggesting deliberate inflation of reported earnings through aggressive accounting choices.

> **CFA Exam Tip:** The Jones model is a cross-sectional regression, which means it requires a panel of firms in the same industry and year. Discretionary accruals are *relative* — a firm is flagged not because its accruals are high in absolute terms, but because they exceed what is normal *for firms of similar size and growth*. This peer-relative approach makes the model robust to industry-level differences in accrual intensity.

The mean discretionary accruals (residuals) are notably higher for the manipulator group. While no individual firm can be convicted on the basis of a Jones model residual alone, the systematic pattern across groups validates the model's ability to separate aggressive from conservative reporting.

## 5. Revenue Manipulation Techniques

Revenue manipulation is the most common form of earnings management because revenue is the
top line of the income statement and has a disproportionate impact on earnings, valuations,
and market sentiment. The following techniques represent the most frequently observed
methods.

### 5.1 Channel Stuffing

**Definition:** Inducing customers (typically distributors or retailers) to purchase more
inventory than they need, usually by offering generous return rights, extended payment
terms, or deep discounts near the end of a reporting period.

**How it works:**
1. Near quarter-end, the company ships excess product to distributors
2. Revenue is recognized upon shipment (FOB shipping point)
3. In the next period, excess inventory is returned or sales collapse because the channel
   is saturated

**Financial statement impact:**
- Revenue and accounts receivable spike in the current quarter
- Next quarter shows weak sales and high returns
- Days Sales Outstanding (DSO) increases because receivables grow faster than genuine demand
- Inventory at the distributor level becomes bloated

**Red flags:**
- Revenue growth concentrated in the last weeks of each quarter
- Rising DSO despite stable or declining industry conditions
- Unusual increase in sales returns in subsequent periods
- Growing gap between revenue growth and cash collection growth

> **Common Mistake:** Students sometimes confuse channel stuffing with legitimate sales
> promotions. The key distinction is whether the customers have genuine demand for the
> product or are being pressured to accept inventory they will likely return. Legitimate
> promotions stimulate end-consumer demand; channel stuffing merely shifts inventory timing.

### 5.2 Bill-and-Hold Arrangements

**Definition:** Recording a sale when the customer has agreed to purchase goods but the
seller retains physical possession of the goods, ostensibly at the customer's request.

**Legitimate use:** A customer may request delayed delivery due to lack of storage space,
and the seller holds the goods in its warehouse. Revenue can be recognized if specific
criteria are met (risks of ownership transfer, delivery schedule is fixed, etc.).

**Manipulative use:** The seller "parks" inventory by recording fictitious or premature
sales to related parties or cooperative customers, never intending to actually deliver the
goods. The most infamous case is Sunbeam Corporation (1996-1998), which used bill-and-hold
to inflate revenue by tens of millions of dollars.

**Red flags:**
- Large bill-and-hold transactions near period-end
- Bill-and-hold revenue as a growing percentage of total revenue
- Customer concentration in bill-and-hold arrangements
- Inventory levels that remain high despite recorded "sales" 

### 5.3 Premature Revenue Recognition

**Definition:** Recording revenue before all recognition criteria are satisfied — before
delivery, before the earnings process is complete, or before collectibility is reasonably
assured.

**Common forms:**
- Recording multi-year contract revenue upfront instead of over the performance period
- Recognizing revenue on percentage-of-completion basis with inflated completion estimates
- Recording software license revenue before delivery and customer acceptance
- Recognizing contingent revenue (dependent on future events) as if it were certain

**Under IFRS 15 / ASC 606**, revenue is recognized when control of goods or services
transfers to the customer. The five-step model requires:
1. Identify the contract
2. Identify performance obligations
3. Determine the transaction price
4. Allocate the price to performance obligations
5. Recognize revenue as obligations are satisfied

Premature recognition typically involves manipulation of step 5 — claiming that
performance obligations are satisfied when they are not.

> **Key Concept:** Under the new revenue recognition standard (IFRS 15 / ASC 606),
> the concept of "control transfer" replaced the older "risks and rewards" model.
> Analysts should pay attention to how companies define their performance obligations
> and when they consider control to have transferred, as these are key judgment areas
> susceptible to manipulation.

### 5.4 Round-Tripping (Circular Transactions)

**Definition:** Two (or more) companies agree to simultaneously purchase from each other,
creating the appearance of revenue without any net economic activity. Each company records
revenue from the other, inflating top-line growth.

**How it works:**
1. Company A "sells" services worth 10 million to Company B
2. Company B simultaneously "sells" services worth 10 million to Company A
3. Both companies report 10 million in additional revenue
4. The transactions may involve actual cash flows (netting to zero) or may be purely
   paper entries

**Famous examples:**
- WorldCom engaged in round-tripping with other telecom companies
- Several energy trading companies in the early 2000s inflated revenue through wash trades

**Red flags:**
- Revenue from related parties or entities with reciprocal business relationships
- Revenue growth that significantly exceeds industry peers
- Cash flows from operations that do not keep pace with revenue growth
- Complex multi-party transactions with unclear economic substance

> **CFA Exam Tip:** When analyzing revenue quality, always compare revenue growth to
> cash collection growth. If revenue is growing at 20% but operating cash flow is flat
> or declining, this is a significant red flag regardless of the specific technique
> being used.

## 6. Expense Manipulation Techniques

While revenue manipulation inflates the top line, expense manipulation works from the
bottom up — reducing or deferring recognized costs to inflate reported earnings. These
techniques are often subtler and harder to detect than revenue manipulation.

### 6.1 Capitalizing vs. Expensing

**Definition:** Recording an expenditure as an asset (capitalized) rather than an expense,
thereby deferring its impact on the income statement to future periods through depreciation
or amortization.

**Legitimate capitalization:** Under GAAP/IFRS, expenditures that provide future economic
benefits beyond the current period should be capitalized. Examples include purchasing
equipment, constructing a building, or developing a software product.

**Manipulative capitalization:** Capitalizing costs that provide no future benefit or that
should clearly be expensed, such as:
- Routine maintenance costs recorded as capital improvements
- Operating expenses reclassified as "development costs"
- Ordinary administrative costs capitalized as "project costs"

**Financial statement impact:**
- Current period: Higher assets, lower expenses, higher net income, lower CFO (capex is
  investing cash flow, not operating)
- Future periods: Higher depreciation/amortization, lower net income

**The classic case:** WorldCom capitalized approximately 3.8 billion in ordinary line
costs (network operating expenses) as capital expenditures, which was one of the largest
accounting frauds in history at the time.

> **Key Concept:** Capitalization manipulation has a distinctive signature in the cash flow
> statement: CFO increases (because the expense moves from operating to investing) while
> free cash flow (CFO minus capex) remains roughly unchanged. This is why analysts should
> always examine free cash flow, not just operating cash flow, when assessing earnings
> quality.

### 6.2 Cookie Jar Reserves

**Definition:** Establishing excessive reserves (provisions) during good years and drawing
them down during bad years to smooth earnings over time.

**How it works:**
1. In a strong year, management records unusually large provisions for bad debts,
   warranties, restructuring, or litigation — more than is economically justified
2. This reduces current earnings but creates a "reserve" on the balance sheet
3. In a weak year, management releases these excess reserves by reducing the provision,
   which flows through as a credit to the income statement, boosting earnings

**Why it matters:**
- Cookie jar reserves make earnings appear more stable and predictable than they really are
- They obscure the true volatility of the underlying business
- They allow management to consistently "meet expectations" even when actual performance
  fluctuates

**Red flags:**
- Unusually large provisions relative to historical averages or industry norms
- Sudden reversals of prior-period provisions that boost current earnings
- Provisions that swing dramatically from year to year without corresponding changes in
  the underlying risk factors

> **Common Mistake:** Cookie jar reserves are sometimes confused with conservative
> accounting. While both involve overstating expenses in the current period, their intent
> is different. Conservative accounting genuinely aims to err on the side of caution.
> Cookie jar reserves are specifically designed to be reversed later, making the intent
> manipulative even if the initial charge might appear conservative.

### 6.3 Big Bath Accounting

**Definition:** Taking massive write-downs and charges in a single period (typically when
a new CEO takes over or when earnings are already going to miss expectations badly),
concentrating all the bad news into one period to "clean the slate."

**Economic motivation:**
- If you are going to miss expectations by a small amount, you might as well miss by a
  large amount — the incremental stock price penalty for a larger miss is small relative
  to the benefit of having a "clean" starting point for future periods
- New CEOs can blame write-downs on their predecessor's decisions
- Future periods benefit from lower asset bases (less depreciation) and absence of the
  problems that were written off

**Financial statement impact:**
- Big bath period: Extremely low earnings, large asset write-downs, high restructuring charges
- Subsequent periods: Artificially improved earnings due to lower cost base

**Red flags:**
- Massive one-time charges coinciding with management changes
- Write-downs that seem disproportionate to the underlying impairment
- Significant earnings improvement in the period immediately following a big bath

### 6.4 Understating Depreciation and Amortization

**Definition:** Using overly optimistic assumptions about the useful life or salvage value
of long-lived assets to reduce annual depreciation expense.

**How it works:**
- Extending the estimated useful life of equipment from 5 years to 10 years cuts annual
  depreciation expense roughly in half
- Increasing the estimated salvage value reduces the depreciable base, similarly reducing
  annual charges
- Changing from an accelerated method (double declining balance) to straight-line reduces
  early-year charges

**Financial statement impact:**
- Lower depreciation expense increases current net income
- Assets remain on the balance sheet at higher carrying values
- If the estimates prove too optimistic, eventual impairments or write-offs are larger

**Red flags:**
- Useful life estimates that exceed industry norms
- Changes in depreciation methods or estimates not accompanied by clear economic justification
- Capital expenditure well below depreciation for extended periods (suggesting assets are not
  being maintained or replaced despite being carried on the books)

> **CFA Exam Tip:** Changes in accounting estimates (like useful lives and salvage values)
> are applied prospectively, not retroactively. This means the impact appears in current
> and future periods but prior periods are not restated. Analysts must carefully read the
> notes to financial statements to identify such changes and assess their impact on
> earnings comparability.

## 7. The Beneish M-Score

### 7.1 Background and Purpose

The **Beneish M-Score** is a mathematical model developed by Professor Messod D. Beneish
(Indiana University) in his 1999 paper *"The Detection of Earnings Manipulation."* The model
uses financial statement data to estimate the probability that a company is manipulating its
reported earnings.

The M-Score combines **eight financial ratios** (indices) that capture different dimensions
of earnings manipulation. Each index compares the current year to the prior year, looking
for unusual changes that are consistent with manipulation patterns observed in known fraud
cases.

> **Key Concept:** The Beneish M-Score is a **probabilistic screening tool**, not a definitive
> fraud detector. A high M-Score indicates that a company's financial profile is statistically
> similar to companies that were later found to have manipulated their earnings. It is designed
> to flag companies for further investigation, not to prove fraud.

### 7.2 The Eight Component Indices

Each index is designed to capture a specific financial signature of earnings manipulation.

---

#### Index 1: Days Sales in Receivables Index (DSRI)

$$\text{DSRI} = \frac{\text{Receivables}_t / \text{Revenue}_t}{\text{Receivables}_{t-1} / \text{Revenue}_{t-1}}$$

**Economic motivation:** A large increase in DSRI suggests that receivables are growing faster
than revenue. This could indicate:
- Revenue inflation through channel stuffing or premature recognition
- Fictitious revenue recorded with corresponding fake receivables
- Loosening of credit standards to pull forward sales

**Interpretation:** DSRI > 1.0 means days sales in receivables increased year-over-year.
Manipulators in Beneish's sample had a mean DSRI of **1.465** versus 1.031 for non-manipulators.

---

#### Index 2: Gross Margin Index (GMI)

$$\text{GMI} = \frac{\text{Gross Margin}_{t-1}}{\text{Gross Margin}_t}$$

where Gross Margin = (Revenue - COGS) / Revenue.

**Economic motivation:** A declining gross margin (GMI > 1) means the company is becoming less
profitable on its core operations. Companies facing margin pressure have stronger incentives
to manipulate earnings to mask deteriorating fundamentals.

**Interpretation:** GMI > 1.0 means gross margins deteriorated. This does not directly indicate
manipulation but indicates heightened **incentive** to manipulate. Manipulators had a mean GMI
of **1.193** versus 1.014 for non-manipulators.

---

#### Index 3: Asset Quality Index (AQI)

$$\text{AQI} = \frac{1 - (CA_t + PPE_t) / TA_t}{1 - (CA_{t-1} + PPE_{t-1}) / TA_{t-1}}$$

where $CA$ = current assets, $PPE$ = net property, plant & equipment, $TA$ = total assets.

**Economic motivation:** Asset quality measures the proportion of total assets that are neither
current assets nor fixed assets — essentially "other" or "soft" assets like intangibles,
deferred charges, and goodwill. An increase in AQI suggests the company may be capitalizing
costs or deferring expenses, inflating the asset base with items of questionable value.

**Interpretation:** AQI > 1.0 means the proportion of soft assets increased. Manipulators had
a mean AQI of **1.254** versus 1.039 for non-manipulators.

---

#### Index 4: Sales Growth Index (SGI)

$$\text{SGI} = \frac{\text{Revenue}_t}{\text{Revenue}_{t-1}}$$

**Economic motivation:** While revenue growth itself is not manipulation, rapidly growing
companies face intense pressure to maintain their growth trajectory. When organic growth
slows, the temptation to artificially sustain it through aggressive accounting increases.
Growth companies also tend to have stretched internal controls that make manipulation easier.

**Interpretation:** SGI > 1.0 means revenue increased. This is an **incentive** indicator,
not a direct measure of manipulation. Manipulators had a mean SGI of **1.607** versus
1.134 for non-manipulators.

#### Index 5: Depreciation Index (DEPI)

$$\text{DEPI} = \frac{\text{Depreciation Rate}_{t-1}}{\text{Depreciation Rate}_t}$$

where Depreciation Rate = Depreciation / (Depreciation + Net PPE).

**Economic motivation:** A DEPI greater than 1 means the rate of depreciation has slowed.
This could indicate that the company is revising useful life estimates upward or switching
to a slower depreciation method — both of which reduce current expenses and inflate earnings.

**Interpretation:** DEPI > 1.0 means depreciation is slowing relative to the asset base.
Manipulators had a mean DEPI of **1.077** versus 1.001 for non-manipulators.

---

#### Index 6: Selling, General & Administrative Expense Index (SGAI)

$$\text{SGAI} = \frac{\text{SGA}_t / \text{Revenue}_t}{\text{SGA}_{t-1} / \text{Revenue}_{t-1}}$$

**Economic motivation:** A disproportionate increase in SGA expenses relative to sales
(SGAI > 1) suggests declining efficiency or increasing overhead. Like GMI, this is
primarily an **incentive** indicator — companies with rising overhead have more motivation
to manipulate earnings to offset the cost pressure.

**Interpretation:** SGAI > 1.0 means SGA expenses are growing faster than revenue.
Manipulators had a mean SGAI of **1.041** versus 1.054 for non-manipulators. Interestingly,
this is one index where manipulators and non-manipulators had similar values.

---

#### Index 7: Leverage Index (LVGI)

$$\text{LVGI} = \frac{(\text{LTD}_t + \text{CL}_t) / \text{TA}_t}{(\text{LTD}_{t-1} + \text{CL}_{t-1}) / \text{TA}_{t-1}}$$

where $LTD$ = long-term debt, $CL$ = current liabilities, $TA$ = total assets.

**Economic motivation:** Increasing leverage (LVGI > 1) means the company is taking on more
debt relative to assets. Higher leverage increases the risk of violating debt covenants,
which creates a powerful incentive to manipulate earnings to maintain required financial
ratios.

**Interpretation:** LVGI > 1.0 means leverage increased. Manipulators had a mean LVGI of
**1.111** versus 1.037 for non-manipulators.

---

#### Index 8: Total Accruals to Total Assets (TATA)

$$\text{TATA} = \frac{\text{Net Income} - \text{CFO}}{\text{Total Assets}}$$

**Economic motivation:** This is the direct measure of the accrual component of earnings.
High TATA means that a large portion of earnings comes from accrual assumptions rather than
cash generation. As discussed in Section 3, high accruals are the most direct indicator of
potential earnings manipulation.

**Interpretation:** Higher TATA indicates more accrual-driven earnings. Manipulators had a
mean TATA of **0.031** versus 0.018 for non-manipulators.

> **CFA Exam Tip:** You should memorize the eight indices and understand the economic
> motivation behind each one. The CFA exam may ask you to identify which index captures
> a particular type of manipulation. Remember: DSRI and TATA are the most direct
> manipulation indicators, while GMI, SGI, and SGAI are primarily incentive indicators.

### 7.3 The Composite M-Score Formula

The eight indices are combined into a single score using a probit (weighted linear) model:

$$M = -4.84 + 0.920 \cdot DSRI + 0.528 \cdot GMI + 0.404 \cdot AQI + 0.892 \cdot SGI$$
$$\quad + 0.115 \cdot DEPI - 0.172 \cdot SGAI + 4.679 \cdot TATA - 0.327 \cdot LVGI$$

**Decision rule:** If $M > -1.78$, the company is flagged as a **likely manipulator**.

The threshold of $-1.78$ was chosen to balance Type I errors (flagging honest companies)
and Type II errors (missing manipulators). Beneish reported that the model correctly
identified approximately **76% of manipulators** in his sample, with a false positive rate
of approximately **17.5%**.

> **Key Concept:** The coefficients reveal which indices matter most. TATA has the largest
> coefficient (4.679), confirming that the accrual component of earnings is the strongest
> single predictor of manipulation. DSRI (0.920) and SGI (0.892) are the next most
> important, reflecting the significance of receivables growth and sales growth pressure.

| Index | Coefficient | Role |
|-------|-------------|------|
| TATA  | +4.679      | Direct manipulation signal (accruals) |
| DSRI  | +0.920      | Direct manipulation signal (receivables) |
| SGI   | +0.892      | Incentive indicator (growth pressure) |
| GMI   | +0.528      | Incentive indicator (margin deterioration) |
| AQI   | +0.404      | Direct manipulation signal (soft assets) |
| DEPI  | +0.115      | Direct manipulation signal (depreciation) |
| SGAI  | -0.172      | Incentive indicator (overhead pressure) |
| LVGI  | -0.327      | Incentive indicator (leverage pressure) |

In [ ]:
# ------------------------------------------------------------------
# Beneish M-Score: Implementation from scratch
# ------------------------------------------------------------------

def compute_mscore(curr, prev):
    '''
    Compute the Beneish M-Score from current and prior year financial data.

    Parameters
    ----------
    curr : dict  - Current year financials
    prev : dict  - Prior year financials

    Required keys: revenue, cogs, receivables, current_assets,
                   ppe_net, total_assets, depreciation, sga,
                   long_term_debt, current_liabilities, net_income, cfo

    Returns
    -------
    dict with each index and the composite M-Score.
    '''
    # 1. DSRI - Days Sales in Receivables Index
    dsr_curr = curr['receivables'] / curr['revenue']
    dsr_prev = prev['receivables'] / prev['revenue']
    DSRI = dsr_curr / dsr_prev

    # 2. GMI - Gross Margin Index
    gm_curr = (curr['revenue'] - curr['cogs']) / curr['revenue']
    gm_prev = (prev['revenue'] - prev['cogs']) / prev['revenue']
    GMI = gm_prev / gm_curr  # note: prior / current

    # 3. AQI - Asset Quality Index
    aq_curr = 1 - (curr['current_assets'] + curr['ppe_net']) / curr['total_assets']
    aq_prev = 1 - (prev['current_assets'] + prev['ppe_net']) / prev['total_assets']
    AQI = aq_curr / aq_prev if aq_prev != 0 else 1.0

    # 4. SGI - Sales Growth Index
    SGI = curr['revenue'] / prev['revenue']

    # 5. DEPI - Depreciation Index
    dep_rate_curr = curr['depreciation'] / (curr['depreciation'] + curr['ppe_net'])
    dep_rate_prev = prev['depreciation'] / (prev['depreciation'] + prev['ppe_net'])
    DEPI = dep_rate_prev / dep_rate_curr if dep_rate_curr != 0 else 1.0

    # 6. SGAI - SGA Expense Index
    sga_ratio_curr = curr['sga'] / curr['revenue']
    sga_ratio_prev = prev['sga'] / prev['revenue']
    SGAI = sga_ratio_curr / sga_ratio_prev if sga_ratio_prev != 0 else 1.0

    # 7. LVGI - Leverage Index
    lev_curr = (curr['long_term_debt'] + curr['current_liabilities']) / curr['total_assets']
    lev_prev = (prev['long_term_debt'] + prev['current_liabilities']) / prev['total_assets']
    LVGI = lev_curr / lev_prev if lev_prev != 0 else 1.0

    # 8. TATA - Total Accruals to Total Assets
    TATA = (curr['net_income'] - curr['cfo']) / curr['total_assets']

    # Composite M-Score
    M = (-4.84
         + 0.920 * DSRI
         + 0.528 * GMI
         + 0.404 * AQI
         + 0.892 * SGI
         + 0.115 * DEPI
         - 0.172 * SGAI
         + 4.679 * TATA
         - 0.327 * LVGI)

    return {
        'DSRI': DSRI, 'GMI': GMI, 'AQI': AQI, 'SGI': SGI,
        'DEPI': DEPI, 'SGAI': SGAI, 'LVGI': LVGI, 'TATA': TATA,
        'M_Score': M
    }


# ------------------------------------------------------------------
# Example: one company with two years of data
# ------------------------------------------------------------------
prev_year = {
    'revenue': 1000, 'cogs': 600, 'receivables': 150,
    'current_assets': 400, 'ppe_net': 350, 'total_assets': 900,
    'depreciation': 50, 'sga': 200, 'long_term_debt': 250,
    'current_liabilities': 180, 'net_income': 120, 'cfo': 110
}

# Current year: shows signs of manipulation
curr_year = {
    'revenue': 1300, 'cogs': 810, 'receivables': 260,  # receivables up 73% vs revenue up 30%
    'current_assets': 480, 'ppe_net': 360, 'total_assets': 1050,
    'depreciation': 48, 'sga': 270, 'long_term_debt': 300,
    'current_liabilities': 220, 'net_income': 165, 'cfo': 105  # NI up, CFO down
}

result = compute_mscore(curr_year, prev_year)

print("Beneish M-Score Analysis -- Example Company")
print("=" * 55)
for key, val in result.items():
    flag = ""
    if key == 'M_Score':
        flag = " *** LIKELY MANIPULATOR ***" if val > -1.78 else " (below threshold)"
    elif key == 'DSRI' and val > 1.465:
        flag = " (above manipulator mean)"
    elif key == 'GMI' and val > 1.193:
        flag = " (above manipulator mean)"
    elif key == 'AQI' and val > 1.254:
        flag = " (above manipulator mean)"
    elif key == 'SGI' and val > 1.607:
        flag = " (above manipulator mean)"
    elif key == 'TATA' and val > 0.031:
        flag = " (above manipulator mean)"
    print(f"  {key:<10} = {val:>8.4f}{flag}")

print(f"\nThreshold: M > -1.78 indicates likely manipulation")

The M-Score implementation above translates each of the eight Beneish indices into a single composite score. Notice how the function takes two periods of financial data — the *change* between periods is what drives most indices, because manipulation typically manifests as an unusual deviation from the prior year's baseline.

The worked example shows both the individual index values and which ones exceed their respective manipulator-group means. When multiple indices simultaneously flash warning signs, the probability of genuine manipulation increases substantially.

> **Key Concept:** The M-Score weights are derived from a probit regression on a sample of known manipulators (SEC enforcement actions) and matched non-manipulators. The large positive coefficient on TATA (4.679) means that high total accruals relative to assets is the single strongest predictor of manipulation — consistent with the accrual anomaly literature. The negative coefficient on SGAI (-0.172) reflects the empirical finding that manipulators tend to *cut* SG&A spending, possibly to artificially boost operating margins.

## 8. M-Score Classification: Synthetic Portfolio Analysis

We now generate a portfolio of 50 synthetic companies with realistic financial data,
compute M-Scores for each, and evaluate the model's classification performance using
a ROC curve.

> **Key Concept:** In practice, the M-Score is used as a screening tool in large portfolios.
> Analysts compute M-Scores for all holdings and focus detailed investigation on companies
> that score above the threshold. The false positive rate means that not every flagged
> company is actually manipulating, but the model significantly narrows the set of companies
> requiring attention.

### Methodology

The synthetic portfolio simulates what an analyst would encounter in practice:

1. **Generate** realistic financial data for 50 companies, with a known fraction designated as "manipulators" (with systematically inflated financial ratios).
2. **Compute** M-Scores for all companies using the Beneish formula.
3. **Classify** each company using the -1.78 threshold.
4. **Evaluate** performance using a confusion matrix and ROC curve.

This controlled experiment lets us measure the model's discriminatory power with known ground truth — something impossible with real-world data where true manipulation status is rarely confirmed.

In [ ]:
# ------------------------------------------------------------------
# Generate 50 synthetic companies
# ------------------------------------------------------------------
n_companies = 50
np.random.seed(SEED)

# Ground truth: 20% are actual manipulators
true_manipulator = rng.random(n_companies) < 0.20

companies = []
for i in range(n_companies):
    is_manip = true_manipulator[i]

    # Prior year (baseline)
    rev_prev = rng.uniform(500, 5000)
    cogs_prev = rev_prev * rng.uniform(0.55, 0.70)
    rec_prev = rev_prev * rng.uniform(0.10, 0.18)
    ca_prev = rev_prev * rng.uniform(0.30, 0.50)
    ppe_prev = rev_prev * rng.uniform(0.25, 0.45)
    ta_prev = rev_prev * rng.uniform(0.80, 1.20)
    dep_prev = ppe_prev * rng.uniform(0.08, 0.14)
    sga_prev = rev_prev * rng.uniform(0.15, 0.25)
    ltd_prev = ta_prev * rng.uniform(0.15, 0.35)
    cl_prev = ta_prev * rng.uniform(0.15, 0.25)
    ni_prev = rev_prev * rng.uniform(0.05, 0.12)
    cfo_prev = ni_prev * rng.uniform(0.85, 1.15)

    prev = {
        'revenue': rev_prev, 'cogs': cogs_prev, 'receivables': rec_prev,
        'current_assets': ca_prev, 'ppe_net': ppe_prev, 'total_assets': ta_prev,
        'depreciation': dep_prev, 'sga': sga_prev, 'long_term_debt': ltd_prev,
        'current_liabilities': cl_prev, 'net_income': ni_prev, 'cfo': cfo_prev
    }

    # Current year: manipulators have distorted ratios
    if is_manip:
        rev_growth = rng.uniform(1.15, 1.50)
        rec_growth = rev_growth * rng.uniform(1.2, 1.8)  # receivables grow faster
        margin_decay = rng.uniform(0.97, 1.02)  # margins may deteriorate
        cfo_ratio = rng.uniform(0.50, 0.85)  # CFO lags NI
        soft_asset_boost = rng.uniform(1.05, 1.20)
        dep_slowdown = rng.uniform(0.85, 0.95)
    else:
        rev_growth = rng.uniform(0.95, 1.15)
        rec_growth = rev_growth * rng.uniform(0.90, 1.10)
        margin_decay = rng.uniform(0.98, 1.02)
        cfo_ratio = rng.uniform(0.90, 1.20)
        soft_asset_boost = rng.uniform(0.95, 1.05)
        dep_slowdown = rng.uniform(0.97, 1.03)

    rev_curr = rev_prev * rev_growth
    cogs_curr = rev_curr * (cogs_prev / rev_prev) / margin_decay
    rec_curr = rec_prev * rec_growth
    ca_curr = ca_prev * rev_growth * rng.uniform(0.95, 1.05)
    ppe_curr = ppe_prev * rng.uniform(1.0, 1.10)
    ta_curr = (ca_curr + ppe_curr) / (1 - (1 - (ca_prev + ppe_prev) / ta_prev) * soft_asset_boost)
    dep_curr = dep_prev * dep_slowdown * (ppe_curr / ppe_prev)
    sga_curr = sga_prev * rev_growth * rng.uniform(0.95, 1.08)
    ltd_curr = ltd_prev * rng.uniform(0.95, 1.15)
    cl_curr = cl_prev * rng.uniform(0.95, 1.10)
    ni_curr = (rev_curr - cogs_curr - dep_curr - sga_curr) * rng.uniform(0.60, 0.80)
    ni_curr = max(ni_curr, 1.0)
    cfo_curr = ni_curr * cfo_ratio

    curr = {
        'revenue': rev_curr, 'cogs': cogs_curr, 'receivables': rec_curr,
        'current_assets': ca_curr, 'ppe_net': ppe_curr, 'total_assets': ta_curr,
        'depreciation': dep_curr, 'sga': sga_curr, 'long_term_debt': ltd_curr,
        'current_liabilities': cl_curr, 'net_income': ni_curr, 'cfo': cfo_curr
    }

    scores = compute_mscore(curr, prev)
    scores['true_manipulator'] = is_manip
    scores['company_id'] = i + 1
    companies.append(scores)

# Extract results
m_scores = np.array([c['M_Score'] for c in companies])
true_labels = np.array([c['true_manipulator'] for c in companies])
predicted = m_scores > -1.78

# Classification results
tp = np.sum(predicted & true_labels)
fp = np.sum(predicted & ~true_labels)
tn = np.sum(~predicted & ~true_labels)
fn = np.sum(~predicted & true_labels)

n_manip = true_labels.sum()
n_clean = (~true_labels).sum()

print("M-Score Classification Results")
print("=" * 50)
print(f"Total companies:        {n_companies}")
print(f"Actual manipulators:    {n_manip}")
print(f"Actual non-manipulators:{n_clean}")
print(f"\nConfusion Matrix (threshold = -1.78):")
print(f"                    Predicted Manip  Predicted Clean")
print(f"  Actual Manip       {tp:>10}       {fn:>10}")
print(f"  Actual Clean       {fp:>10}       {tn:>10}")
if (tp + fn) > 0:
    print(f"\nSensitivity (TPR):   {tp/(tp+fn):.2%}")
if (tn + fp) > 0:
    print(f"Specificity (TNR):   {tn/(tn+fp):.2%}")
if (tp + fp) > 0:
    print(f"Precision:           {tp/(tp+fp):.2%}")

The confusion matrix reveals the model's classification performance at the standard threshold of $M = -1.78$:

* **True positives** — manipulators correctly identified. These are the wins that justify running the model.
* **False positives** — clean companies incorrectly flagged. These waste investigative effort but cause no harm beyond that.
* **False negatives** — manipulators missed. These are the costly errors — the model's blind spots.
* **True negatives** — clean companies correctly cleared.

> **Common Mistake:** Students often focus solely on accuracy (total correct / total). But in fraud detection, **sensitivity** (catching actual manipulators) matters far more than overall accuracy, because the base rate of manipulation is low. A model that classifies every company as "clean" would have high accuracy but zero sensitivity — completely useless for its intended purpose.

In [ ]:
# ------------------------------------------------------------------
# ROC Curve
# ------------------------------------------------------------------
# Compute ROC by sweeping thresholds
thresholds = np.linspace(m_scores.min() - 1, m_scores.max() + 1, 500)
tpr_list = []
fpr_list = []

for thresh in thresholds:
    pred = m_scores > thresh
    tpr = np.sum(pred & true_labels) / max(np.sum(true_labels), 1)
    fpr = np.sum(pred & ~true_labels) / max(np.sum(~true_labels), 1)
    tpr_list.append(tpr)
    fpr_list.append(fpr)

tpr_arr = np.array(tpr_list)
fpr_arr = np.array(fpr_list)

# AUC via trapezoidal rule (sort by FPR)
sorted_idx = np.argsort(fpr_arr)
auc = np.trapz(tpr_arr[sorted_idx], fpr_arr[sorted_idx])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
axes[0].plot(fpr_arr, tpr_arr, color=PRIMARY, linewidth=2, label=f"M-Score ROC (AUC={auc:.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random classifier")
# Mark the -1.78 threshold
thresh_idx = np.argmin(np.abs(thresholds - (-1.78)))
axes[0].scatter([fpr_arr[thresh_idx]], [tpr_arr[thresh_idx]], color=SECONDARY, s=100, zorder=5,
                label=f"Threshold = -1.78")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve: M-Score Classification")
axes[0].legend(loc="lower right")
axes[0].set_xlim(-0.02, 1.02)
axes[0].set_ylim(-0.02, 1.02)

# M-Score distribution
bins = np.linspace(m_scores.min() - 0.5, m_scores.max() + 0.5, 20)
axes[1].hist(m_scores[~true_labels], bins=bins, alpha=0.7, color=PRIMARY, label="Non-manipulators")
axes[1].hist(m_scores[true_labels], bins=bins, alpha=0.7, color=SECONDARY, label="Manipulators")
axes[1].axvline(-1.78, color="black", linewidth=2, linestyle="--", label="Threshold = -1.78")
axes[1].set_xlabel("M-Score")
axes[1].set_ylabel("Count")
axes[1].set_title("M-Score Distribution by True Label")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"AUC: {auc:.3f}")
print(f"A perfect classifier would have AUC = 1.0; random = 0.5")

The ROC curve plots the trade-off between sensitivity (catching true manipulators) and specificity (not falsely accusing clean companies) across all possible thresholds. An AUC significantly above 0.5 confirms that the M-Score contains genuine discriminatory power — it separates manipulators from non-manipulators better than random chance.

The histogram of M-Score distributions shows the key diagnostic: manipulator scores are shifted to the right (higher values) relative to clean companies, though there is substantial overlap in the middle range. This overlap is why no single threshold can perfectly separate the two groups.

> **CFA Exam Tip:** The AUC (Area Under the ROC Curve) is a threshold-independent measure of classifier quality. An AUC of 0.80 means that a randomly chosen manipulator will have a higher M-Score than a randomly chosen clean company 80% of the time. Higher AUC = better model discrimination.

In [ ]:
# ------------------------------------------------------------------
# Detailed look at flagged companies
# ------------------------------------------------------------------
print("Companies Flagged as Likely Manipulators (M > -1.78)")
print("=" * 90)
header = f"{'ID':>4} {'M-Score':>8} {'DSRI':>7} {'GMI':>7} {'AQI':>7} {'SGI':>7} {'TATA':>7} {'Actual':>10}"
print(header)
print("-" * 90)

for c in sorted(companies, key=lambda x: -x['M_Score']):
    if c['M_Score'] > -1.78:
        actual = "MANIP" if c['true_manipulator'] else "Clean"
        print(f"{c['company_id']:>4} {c['M_Score']:>8.3f} {c['DSRI']:>7.3f} {c['GMI']:>7.3f} "
              f"{c['AQI']:>7.3f} {c['SGI']:>7.3f} {c['TATA']:>7.4f} {actual:>10}")

The detailed breakdown of flagged companies reveals which M-Score components are driving the classification. Companies with extreme values in multiple indices — particularly DSRI (receivables growing faster than revenue), TATA (high total accruals), and SGI (rapid sales growth) — cluster at the top of the suspicion list.

> **Key Concept:** A single elevated index is not necessarily damning — a company could have a high Sales Growth Index simply because it is in a hypergrowth phase. The power of the M-Score lies in the *combination* of signals. Manipulation typically manifests across multiple indices simultaneously because the underlying distortions ripple through the entire set of financial statements.

Note the false positives: some flagged companies are genuinely clean. This is the inherent trade-off in any screening model — a lower threshold catches more manipulators but also flags more honest companies. In practice, the M-Score is a *triage tool* that narrows the investigation list, not a definitive verdict.

## 9. Red Flags Checklist

### 9.1 Income Statement Red Flags

A comprehensive list of warning signs that analysts should watch for when assessing earnings
quality from the income statement:

1. **Revenue growth significantly exceeding industry peers** — may indicate aggressive
   recognition or channel stuffing
2. **Gross margins trending in opposite direction from peers** — suggests cost misclassification
   or revenue inflation
3. **Unusual one-time gains** used to meet earnings targets — asset sales, pension gains,
   litigation settlements timed to offset operating shortfalls
4. **Frequent "non-recurring" charges** that recur every year — restructuring charges,
   asset impairments, and write-downs that management excludes from "adjusted" earnings
   but that represent real economic costs
5. **Changes in revenue recognition policies** without clear business justification
6. **Growing gap between GAAP earnings and "adjusted" or "pro forma" earnings** — management
   may be directing attention away from unfavorable GAAP results
7. **Earnings that consistently meet or beat consensus by a penny** — statistically unlikely
   pattern that suggests active management of results

> **Common Mistake:** Analysts sometimes dismiss one-time gains as irrelevant. While they
> may not be recurring, the timing of their recognition is often discretionary. A company
> that consistently uses one-time gains to meet targets is engaging in earnings management
> even if each individual transaction is legitimate.

### 9.2 Balance Sheet Red Flags

1. **Accounts receivable growing faster than revenue** — the single most reliable indicator
   of revenue manipulation (captured by DSRI in the M-Score)
2. **Inventory growing faster than cost of goods sold** — may indicate obsolete inventory
   not being written down, or channel stuffing in reverse (building inventory for future
   fictitious sales)
3. **Rapid growth in "other assets" or intangible assets** — may indicate improper
   capitalization of expenses
4. **Declining allowance for doubtful accounts as a percentage of receivables** — suggests
   the company is not adequately reserving for bad debts
5. **Off-balance-sheet arrangements** — special purpose entities, operating leases (pre-IFRS 16),
   unconsolidated subsidiaries that hide leverage and risk
6. **Related-party transactions** — transactions with entities controlled by management or
   board members that may not be at arm's length

### 9.3 Cash Flow Statement Red Flags

1. **Net income significantly exceeding CFO for multiple periods** — the most fundamental
   indicator of accrual-driven earnings
2. **CFO declining while net income increases** — a divergence that almost always warrants
   investigation
3. **Capitalizing operating expenses** — shifting cash outflows from operating to investing
   activities, making CFO appear stronger than it is
4. **Unusual cash flows from financing activities** — factoring receivables, securitizing
   assets, or using supply chain finance to reclassify operating outflows
5. **Free cash flow consistently below dividends or buybacks** — the company is borrowing
   or selling assets to fund shareholder distributions, which is unsustainable

> **Key Concept:** The cash flow statement is the analyst's best friend when assessing
> earnings quality. While the income statement reflects management's estimates and judgments,
> the cash flow statement reflects actual cash transactions. Persistent divergences between
> the two are the most reliable indicator of earnings manipulation.

### 9.4 Qualitative Red Flags

Beyond the numbers, certain qualitative factors should heighten an analyst's suspicion:

1. **Frequent changes in auditors** — may indicate disagreements about accounting treatments
2. **Restatements of prior period financials** — prior manipulation coming to light
3. **High management turnover**, especially in the CFO position
4. **Aggressive executive compensation** tied primarily to short-term earnings targets
5. **Weak corporate governance** — combined CEO/Chair, lack of independent directors,
   weak audit committee
6. **Complex corporate structure** with numerous subsidiaries, special purpose entities,
   and intercompany transactions
7. **Management that is hostile to analyst questions** or provides evasive answers about
   accounting policies
8. **Insider selling** by executives while publicly expressing confidence in the company

> **CFA Exam Tip:** The CFA curriculum groups red flags into three categories:
> (1) revenue recognition issues, (2) expense recognition issues, and (3) cash flow
> versus earnings divergences. Be prepared to identify which category a specific red
> flag falls into and explain why it is concerning.

## 10. The Auditor's Role and Limitations

### 10.1 What an Audit Is

A financial statement audit is an independent examination of a company's financial statements
to determine whether they are presented fairly, in all material respects, in accordance with
the applicable financial reporting framework (GAAP or IFRS).

**Key features of an audit:**
- **Reasonable assurance, not absolute assurance.** An audit is designed to detect material
  misstatements, but it cannot guarantee that all fraud or errors will be found.
- **Materiality threshold.** Auditors focus on items that are large enough to influence the
  decisions of reasonable users of the financial statements. Small misstatements may go
  undetected or uncorrected.
- **Sampling-based.** Auditors do not examine every transaction. They use statistical and
  judgmental sampling to draw conclusions about the population of transactions.
- **Based on evidence.** Auditors gather audit evidence through inspection, observation,
  inquiry, confirmation, recalculation, and analytical procedures.

### 10.2 What Auditors Can Detect

Auditors are reasonably effective at detecting:
- **Mathematical errors** and posting mistakes
- **Violations of specific accounting standards** where the rules are clear
- **Material inconsistencies** between financial statements and supporting documentation
- **Going concern risks** based on financial ratios and cash flow analysis
- **Related-party transactions** that are not properly disclosed
- **Control weaknesses** in the company's internal control environment

### 10.3 What Auditors Struggle to Detect

Audits have inherent limitations in detecting:
- **Management fraud involving collusion** — when multiple executives conspire to falsify
  records, the auditor may not be able to detect it through normal procedures
- **Sophisticated revenue manipulation** — channel stuffing involving real shipments to
  real customers is difficult to distinguish from legitimate sales
- **Estimates that are biased but within the range of reasonable** — if management
  consistently chooses the most aggressive end of acceptable ranges, each individual
  estimate may pass audit scrutiny even though the cumulative effect is material
- **Transactions with no paper trail** — side agreements, verbal commitments, and
  undocumented arrangements are invisible to auditors who rely on documentation
- **Fraud by those who control the audit relationship** — management selects and pays
  the auditor, creating an inherent conflict of interest despite independence requirements

> **Key Concept:** The "expectation gap" refers to the difference between what the public
> believes an audit provides (a guarantee that the financial statements are correct and the
> company is well-managed) and what an audit actually provides (reasonable assurance that the
> financial statements are free from material misstatement). This gap is a persistent issue
> in the accounting profession.

### 10.4 The Analyst's Independent Role

Because of audit limitations, **financial analysts must perform their own earnings quality
assessment** rather than relying solely on the auditor's opinion. The tools covered in this
notebook — accrual analysis, the M-Score, red flag identification — are part of the
analyst's independent toolkit for assessing whether reported earnings faithfully represent
economic reality.

> **CFA Exam Tip:** The CFA Institute emphasizes that a clean audit opinion does not
> guarantee the absence of earnings manipulation. Enron, WorldCom, and many other major
> fraud cases received clean audit opinions in the years immediately preceding their
> collapse. Analysts must exercise independent judgment regardless of the auditor's
> conclusions.

### 10.5 Practical Implications for Investors

When evaluating a company's financial statements, investors should:

1. **Read the auditor's report carefully** — look for qualifications, emphasis of matter
   paragraphs, and key audit matters that highlight areas of significant judgment
2. **Review the notes to the financial statements** — accounting policy choices, significant
   estimates, and related-party disclosures often reveal more than the face of the statements
3. **Compare financial statements across multiple years** — one year of aggressive accounting
   may not be obvious, but a pattern of increasingly aggressive choices is much easier to detect
4. **Benchmark against peers** — ratios that deviate significantly from industry norms
   deserve investigation
5. **Use quantitative tools like the M-Score** as a systematic screen before diving into
   detailed analysis

> **Common Mistake:** Investors sometimes assume that because a large, reputable audit firm
> conducted the audit, the financial statements must be reliable. The Big Four firms audited
> Enron (Arthur Andersen), Wirecard (EY), Luckin Coffee (EY), and numerous other companies
> that were later found to have committed fraud. The reputation of the auditor provides some
> comfort but is not a substitute for independent analysis.

### Practical workflow for the earnings quality analyst

A structured approach to earnings quality assessment:

1. **Screen** — Run the Beneish M-Score (or similar quantitative model) across the portfolio to identify companies above the threshold.
2. **Investigate** — For flagged companies, examine the specific indices driving the score. Which accrual categories are abnormal?
3. **Compare** — Benchmark the company's accruals, margins, and growth against industry peers. Is it an outlier?
4. **Read footnotes** — Accounting policy changes, related-party transactions, off-balance-sheet items, and auditor reports often contain critical context.
5. **Assess materiality** — Even if manipulation exists, is it large enough to affect the investment thesis?
6. **Document** — Record findings and the basis for conclusions. Earnings quality assessment is inherently judgmental and benefits from a clear audit trail.

> **Key Concept:** Earnings quality analysis is fundamentally about asking the right questions, not about applying a mechanical formula. Quantitative tools like the M-Score and accrual ratios narrow the field of inquiry; the analyst's judgment completes the assessment.

## 11. References

1. **Sloan, R. G.** (1996). "Do Stock Prices Fully Reflect Information in Accruals and Cash
   Flows about Future Earnings?" *The Accounting Review*, 71(3), 289-315.

2. **Beneish, M. D.** (1999). "The Detection of Earnings Manipulation." *Financial Analysts
   Journal*, 55(5), 24-36.

3. **Jones, J. J.** (1991). "Earnings Management During Import Relief Investigations."
   *Journal of Accounting Research*, 29(2), 193-228.

4. **Dechow, P. M., Sloan, R. G., & Sweeney, A. P.** (1995). "Detecting Earnings Management."
   *The Accounting Review*, 70(2), 193-225.

5. **Dechow, P. M., Ge, W., & Schrand, C.** (2010). "Understanding Earnings Quality: A Review
   of the Proxies, Their Determinants and Their Consequences." *Journal of Accounting and
   Economics*, 50(2-3), 344-401.

6. **CFA Institute** (2024). *CFA Program Curriculum Level I*, Volume 3: Financial Statement
   Analysis. Chapters on Financial Reporting Quality.

7. **Schilit, H. M., & Perler, J.** (2010). *Financial Shenanigans: How to Detect Accounting
   Gimmicks and Fraud in Financial Reports*. 3rd Edition, McGraw-Hill.

8. **Penman, S. H.** (2013). *Financial Statement Analysis and Security Valuation*. 5th Edition,
   McGraw-Hill.

---

*Notebook created for educational purposes. All financial data is synthetic. The Beneish
M-Score is a screening tool and should not be used as the sole basis for investment decisions
or fraud accusations.*

### Further reading for CFA candidates

The CFA Level 1 curriculum covers earnings quality primarily through the "Financial Reporting Quality" and "Financial Analysis Techniques" readings. The key testable concepts are:

* **Spectrum of financial reporting quality** — from GAAP-compliant and decision-useful (highest quality) to fraudulent reporting (lowest quality)
* **Conservative vs aggressive accounting** — understanding the continuum and being able to identify where a company falls
* **Accrual-based signals** — using the CFO-to-NI relationship as a primary diagnostic
* **Beneish M-Score** — understanding the components, composite formula, and interpretation (you will not be asked to compute all 8 indices from scratch, but you should know what each measures)
* **Red flags** — being able to identify warning signs across all three financial statements

The deeper quantitative material (Jones model estimation, ROC curves) extends beyond the CFA L1 curriculum but provides the analytical foundation for understanding *why* the simpler heuristics work.

> **Final note:** Mastery of the concepts in this notebook is essential for the CFA Level 1 exam, as well as for practical financial analysis work. Practice the worked examples by hand and verify your understanding by reproducing the code from scratch.
